In [31]:
from __future__ import print_function

import sys
if sys.version_info >= (3,0) :
        int()
from pyspark.sql import SparkSession

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.sql import Row

from pyspark.sql.functions import col, isnan, when, count,trim

In [32]:
spark = SparkSession\
    .builder\
    .appName("Pratica ALS")\
    .getOrCreate()

In [33]:
lines = spark.read.text("Exemplo.txt").rdd
filtered_lines = lines.filter(lambda row: not row.value.startswith("userId"))
parts = filtered_lines.map(lambda row: row.value.split(","))
ratingsRDD = parts.map(lambda p: Row(userId=int(p[0]), movieId=int(p[1]),
                                    rating=float(p[2]),timestamp=int(p[3])))
ratings = spark.createDataFrame(ratingsRDD)

In [34]:
ratings.show(30)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|     10|   4.0|964982703|
|     1|     20|   5.0|964981247|
|     1|     30|   3.0|964982224|
|     2|     10|   2.0|964982931|
|     2|     40|   4.0|964983815|
|     2|     50|   5.0|964982400|
|     3|     20|   3.0|964982931|
|     3|     30|   4.0|964983815|
|     3|     60|   2.0|964982400|
|     4|     70|   5.0|964982703|
|     4|     80|   3.0|964981247|
|     4|     90|   4.0|964982224|
|     5|    100|   5.0|964982931|
|     5|    110|   2.0|964983815|
|     5|    120|   4.0|964982400|
|     6|    130|   3.0|964982703|
|     6|    140|   5.0|964981247|
|     6|    150|   4.0|964982224|
|     7|    160|   2.0|964982931|
|     7|    170|   5.0|964983815|
|     7|    180|   3.0|964982400|
|     8|    190|   4.0|964982703|
|     8|    200|   5.0|964981247|
|     8|    210|   2.0|964982224|
|     9|    220|   3.0|964982931|
|     9|    230|   4.0|964983815|
|     9|    24

In [44]:
(training,test) = ratings.randomSplit([0.8, 0.2])

In [45]:
training_clean = training.filter(
    # Remover nulos
    col("userId").isNotNull() & 
    col("movieId").isNotNull() & 
    col("rating").isNotNull() &
    # Remover strings vazias e espaços em branco
    (trim(col("userId")) != "") &
    (trim(col("movieId")) != "") &
    (trim(col("rating")) != "") &
    # Verificar se rating não é NaN
    ~isnan(col("rating"))
)
# Converter para tipos corretos com tratamento de erro
training_clean = training_clean.select(
    col("userId").cast("integer").alias("userId"),
    col("movieId").cast("integer").alias("movieId"), 
    col("rating").cast("double").alias("rating")
).filter(
    # Remover registros onde a conversão falhou (resultou em null)
    col("userId").isNotNull() &
    col("movieId").isNotNull() &
    col("rating").isNotNull() &
    (col("rating") > 0)  # Garantir ratings positivos
)

In [46]:
als = ALS(maxIter=5, regParam=0.1, userCol="userId", itemCol="movieId", ratingCol="rating",
          coldStartStrategy="drop")
model = als.fit(training)

In [47]:
predictions = model.transform(test)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",
                                predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))

Root-mean-square error = 2.3787612915039062


In [53]:
userRecs = model.recommendForAllUsers(10)

In [54]:
userRecs.show()

[Stage 347:===========================================>         (82 + 10) / 100]

+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                                                                                                     |
+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|20    |[{550, 3.9266956}, {440, 2.8975182}, {430, 2.3180149}, {570, 1.9633478}, {140, 1.8180252}, {150, 1.4544202}, {680, 1.4337566}, {670, 1.1470053}, {130, 1.0908152}, {500, 0.7989152}]|
|10    |[{270, 3.924426}, {260, 2.94332}, {520, 2.174653}, {250, 1.962213}, {680, 1.8034325}, {50, 1.6702484}, {360, 1.61221}, {670, 1.4427462}, {240, 1.420043}, {40, 1.3361988}]          |
|1     |[{20, 4.8954577}, {10, 3.9334736}, {30, 3.

In [51]:
movieRecs = model.recommendForAllItems(10)

In [52]:
movieRecs.show()

[Stage 318:==================================================>   (93 + 7) / 100]

+-------+--------------------+
|movieId|     recommendations|
+-------+--------------------+
|    540|[{19, 2.966275}, ...|
|    580|[{21, 2.966275}, ...|
|    300|[{11, 2.966275}, ...|
|    460|[{17, 2.965643}, ...|
|    350|[{13, 3.9541903},...|
|    330|[{12, 1.9747257},...|
|    190|[{8, 3.962315}, {...|
|    360|[{13, 4.9427385},...|
|    140|[{6, 4.942738}, {...|
|    660|[{23, 2.9556675},...|
|    280|[{11, 4.9437914},...|
|    440|[{16, 4.9528937},...|
|     20|[{1, 4.8954577}, ...|
|     40|[{2, 3.944868}, {...|
|    470|[{17, 3.9541903},...|
|    500|[{18, 2.9196877},...|
|    340|[{13, 2.9656425},...|
|    250|[{10, 1.962213}, ...|
|    570|[{20, 1.9633478},...|
|    120|[{5, 3.962315}, {...|
+-------+--------------------+
only showing top 20 rows


In [55]:
users = ratings.select(als.getUserCol()).distinct()

In [57]:
users.show()

+------+
|userId|
+------+
|    19|
|    22|
|     7|
|    25|
|     6|
|     9|
|    17|
|     5|
|     1|
|    10|
|     3|
|    12|
|     8|
|    11|
|     2|
|     4|
|    13|
|    18|
|    14|
|    21|
+------+
only showing top 20 rows


In [59]:
userRecsOnlyItemId = userRecs.select(userRecs['userId'], userRecs['recommendations']['movieId'])

In [62]:
userRecsOnlyItemId.show(10,False)

+------+--------------------------------------------------+
|userId|recommendations.movieId                           |
+------+--------------------------------------------------+
|20    |[550, 440, 430, 570, 140, 150, 680, 670, 130, 500]|
|10    |[270, 260, 520, 250, 680, 50, 360, 670, 240, 40]  |
|1     |[20, 10, 30, 600, 60, 580, 50, 280, 40, 710]      |
|21    |[600, 580, 20, 630, 200, 710, 520, 10, 620, 190]  |
|11    |[280, 300, 400, 100, 120, 140, 420, 150, 410, 20] |
|12    |[320, 310, 330, 240, 220, 370, 680, 50, 70, 670]  |
|22    |[630, 620, 600, 200, 580, 190, 400, 520, 240, 140]|
|2     |[50, 40, 10, 710, 70, 270, 700, 90, 140, 260]     |
|13    |[360, 350, 640, 340, 240, 480, 660, 470, 100, 400]|
|3     |[20, 600, 60, 30, 10, 580, 170, 710, 630, 70]     |
+------+--------------------------------------------------+
only showing top 10 rows


50 recomendacoes para todos os usuarios

In [63]:
userRecs = model.recommendForAllUsers(50)

In [64]:
userRecs.show()

+------+--------------------+
|userId|     recommendations|
+------+--------------------+
|    20|[{550, 3.9266956}...|
|    10|[{270, 3.924426},...|
|     1|[{20, 4.8954577},...|
|    21|[{600, 4.943792},...|
|    11|[{280, 4.9437914}...|
|    12|[{320, 4.9368143}...|
|    22|[{630, 3.9401658}...|
|     2|[{50, 4.9310846},...|
|    13|[{360, 4.9427385}...|
|     3|[{20, 2.9951813},...|
|    23|[{640, 4.926112},...|
|     4|[{70, 4.942738}, ...|
|    24|[{680, 4.9368143}...|
|    14|[{320, 2.277256},...|
|     5|[{100, 4.952894},...|
|    15|[{400, 4.926112},...|
|    25|[{710, 3.9401655}...|
|     6|[{140, 4.942738},...|
|    16|[{440, 4.9528937}...|
|    17|[{480, 4.942738},...|
+------+--------------------+
only showing top 20 rows


In [70]:
!pip install pymongo

In [69]:
recomendationToSave = model.recommendForAllUsers(10)
recomendationToSave = recomendationToSave.select(recomendationToSave['userId'], recomendationToSave['recommendations']['movieId'].alias('movieId'))

In [75]:
import pymongo
from pymongo import MongoClient

client = MongoClient('localhost',27017)
db = client.test

In [77]:
colecao = recomendationToSave.collect()
for row in colecao:
    print(row.asDict())
    db.suggestions.insert_one(row.asDict())

{'userId': 20, 'movieId': [550, 440, 430, 570, 140, 150, 680, 670, 130, 500]}
{'userId': 10, 'movieId': [270, 260, 520, 250, 680, 50, 360, 670, 240, 40]}
{'userId': 1, 'movieId': [20, 10, 30, 600, 60, 580, 50, 280, 40, 710]}
{'userId': 21, 'movieId': [600, 580, 20, 630, 200, 710, 520, 10, 620, 190]}
{'userId': 11, 'movieId': [280, 300, 400, 100, 120, 140, 420, 150, 410, 20]}
{'userId': 12, 'movieId': [320, 310, 330, 240, 220, 370, 680, 50, 70, 670]}
{'userId': 22, 'movieId': [630, 620, 600, 200, 580, 190, 400, 520, 240, 140]}
{'userId': 2, 'movieId': [50, 40, 10, 710, 70, 270, 700, 90, 140, 260]}
{'userId': 13, 'movieId': [360, 350, 640, 340, 240, 480, 660, 470, 100, 400]}
{'userId': 3, 'movieId': [20, 600, 60, 30, 10, 580, 170, 710, 630, 70]}
{'userId': 23, 'movieId': [640, 660, 360, 200, 350, 480, 170, 520, 70, 190]}
{'userId': 4, 'movieId': [70, 90, 80, 710, 170, 680, 700, 640, 670, 180]}
{'userId': 24, 'movieId': [680, 670, 70, 170, 690, 480, 90, 270, 470, 440]}
{'userId': 14, 'mov

In [79]:
recomendationToSave.show()

+------+--------------------+
|userId|             movieId|
+------+--------------------+
|    20|[550, 440, 430, 5...|
|    10|[270, 260, 520, 2...|
|     1|[20, 10, 30, 600,...|
|    21|[600, 580, 20, 63...|
|    11|[280, 300, 400, 1...|
|    12|[320, 310, 330, 2...|
|    22|[630, 620, 600, 2...|
|     2|[50, 40, 10, 710,...|
|    13|[360, 350, 640, 3...|
|     3|[20, 600, 60, 30,...|
|    23|[640, 660, 360, 2...|
|     4|[70, 90, 80, 710,...|
|    24|[680, 670, 70, 17...|
|    14|[320, 370, 310, 2...|
|     5|[100, 120, 400, 2...|
|    15|[400, 100, 420, 2...|
|    25|[710, 70, 170, 70...|
|     6|[140, 150, 130, 2...|
|    16|[440, 430, 550, 1...|
|    17|[480, 470, 460, 5...|
+------+--------------------+
only showing top 20 rows
